# KD smoke test on Kaggle — off-policy vs on-policy

Teacher **Qwen3-4B** → student **Qwen3-0.6B**, full training (no LoRA), on GSM8K.

Goal of this notebook: check that both training modes run, and measure **time per step** and **memory**.
It is not the real experiment yet (only a few steps).

**Setup:** Accelerator = **GPU T4 x2** (both GPUs are used), Internet = on.

**Why two GPUs:** teacher 4B in fp16 needs ~8 GB; full training of the 0.6B student needs ~12 GB
(weights + gradients + AdamW state). Together they don't fit on one 16 GB T4,
so the teacher sits on GPU 1 and the student trains on GPU 0.

**One knob, same idea as GKD's `lmbda`:**
- `LMBDA = 0.0` → off-policy: the student is trained on fixed answers (the dataset solution), matching the teacher's token probabilities on them (Hinton-style soft targets, per token).
- `LMBDA = 1.0` → on-policy: the student writes its own answer, the teacher scores every token of it.

In [1]:
!pip install -q -U transformers datasets accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation; must be set before CUDA starts

import torch
assert torch.cuda.device_count() >= 2, "Select 'GPU T4 x2' in the notebook settings"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {round(p.total_memory / 1e9, 1)} GB, compute capability {p.major}.{p.minor}")

GPU 0: Tesla T4, 15.6 GB, compute capability 7.5
GPU 1: Tesla T4, 15.6 GB, compute capability 7.5


## Config

In [3]:
TEACHER = "Qwen/Qwen3-4B"
STUDENT = "Qwen/Qwen3-0.6B"
TEACHER_DEVICE = "cuda:1"
STUDENT_DEVICE = "cuda:0"

ARMS = [0.0, 1.0]        # LMBDA values to run: 0.0 = off-policy, 1.0 = on-policy
N_TRAIN = 200            # GSM8K train questions available to the smoke test
STEPS = 15               # optimizer steps per arm (smoke test only)
ACCUM = 4                # questions per optimizer step (gradient accumulation)
LR = 1e-5

KL_DIRECTION = "forward" # "forward" = KL(teacher || student), Hinton; "reverse" = KL(student || teacher), Thinking Machines
KD_TEMPERATURE = 1.0     # Hinton's softmax temperature for the distillation loss

# On-policy sampling (Qwen3 recommended non-thinking settings)
MAX_NEW_TOKENS = 384
GEN_KWARGS = dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20)

N_EVAL = 20              # quick GSM8K test check before/after each arm (0 = skip)
INSTRUCTION = "Solve the problem step by step. Put the final number at the end."
SEED = 42

# T4 (compute capability 7.5) has no native bf16 -> fp16
HALF = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("half precision dtype:", HALF)

half precision dtype: torch.float16


## Helpers

In [4]:
import gc, json, random, re, time
from pathlib import Path

import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

random.seed(SEED)
torch.manual_seed(SEED)


def gb(num_bytes):
    return round(num_bytes / 1024**3, 2)


def vram_used_gb(device):
    """Whole-GPU memory in use on one device (same as nvidia-smi)."""
    free, total = torch.cuda.mem_get_info(device)
    return gb(total - free)


def extract_last_number(text):
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(numbers[-1]) if numbers else None


def gold_answer(answer_field):
    return extract_last_number(answer_field.split("####")[-1])


def clean_solution(answer_field):
    """GSM8K solution -> plain text: drop calculator notes <<...>> and rewrite '#### X'."""
    text = re.sub(r"<<[^>]*>>", "", answer_field)
    body, final = text.split("####")
    return f"{body.strip()}\nThe final answer is {final.strip()}."


tokenizer = AutoTokenizer.from_pretrained(STUDENT)
END_TOKEN = "<|im_end|>"  # Qwen chat end-of-turn token


def prompt_ids(question):
    """Chat prompt with thinking OFF, as token ids (list)."""
    messages = [{"role": "user", "content": f"{question}\n\n{INSTRUCTION}"}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    return tokenizer(text, add_special_tokens=False)["input_ids"]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

## Data and teacher

In [5]:
train_set = load_dataset("openai/gsm8k", "main", split="train").select(range(N_TRAIN))
test_set = load_dataset("openai/gsm8k", "main", split="test")

teacher = AutoModelForCausalLM.from_pretrained(TEACHER, dtype=HALF).to(TEACHER_DEVICE)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)
print("teacher loaded | VRAM GPU1:", vram_used_gb(TEACHER_DEVICE), "GB")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

... (4 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

teacher loaded | VRAM GPU1: 7.6 GB


## Quick eval (same grading as eval_gsm8k.py, greedy)

In [6]:
@torch.no_grad()
def quick_eval(model, device, n=N_EVAL):
    if n == 0:
        return None
    model.eval()
    correct = 0
    for example in test_set.select(range(n)):
        ids = torch.tensor([prompt_ids(example["question"])], device=device)
        with torch.autocast("cuda", dtype=HALF):
            out = model.generate(ids, max_new_tokens=512, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        response = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
        predicted = extract_last_number(response)
        correct += predicted is not None and abs(predicted - gold_answer(example["answer"])) < 1e-6
    return correct / n

## Distillation loss (per token)

In [7]:
def token_kl(student_logits, log_p_t):
    """Mean per-token KL between teacher and student distributions over the response tokens."""
    log_p_s = F.log_softmax(student_logits.float() / KD_TEMPERATURE, dim=-1)
    n_tokens = log_p_s.shape[0] * log_p_s.shape[1]
    if KL_DIRECTION == "forward":   # KL(teacher || student): student must cover what the teacher considers likely
        kl = F.kl_div(log_p_s, log_p_t, reduction="sum", log_target=True)
    else:                           # KL(student || teacher): student is punished where it puts mass the teacher dislikes
        kl = (log_p_s.exp() * (log_p_s - log_p_t)).sum()
    return kl / n_tokens * KD_TEMPERATURE**2   # T^2 keeps gradient scale comparable across temperatures (Hinton)


def distill_loss(student, full, n_prompt, vocab):
    """Teacher scores the response on its own GPU; only its log-probs travel to the student GPU."""
    with torch.no_grad():
        t_logits = teacher(full.to(TEACHER_DEVICE)).logits[:, n_prompt - 1:-1, :vocab]
        log_p_t = F.log_softmax(t_logits.float() / KD_TEMPERATURE, dim=-1).to(STUDENT_DEVICE)
        del t_logits
    with torch.autocast("cuda", dtype=HALF):
        s_logits = student(full, use_cache=False).logits[:, n_prompt - 1:-1, :vocab]
    loss = token_kl(s_logits, log_p_t)
    del s_logits, log_p_t
    return loss

## Training one arm

In [8]:
MAX_PROMPT = max(len(prompt_ids(q)) for q in train_set["question"])
WORST_CASE_LEN = MAX_PROMPT + MAX_NEW_TOKENS
print("longest prompt:", MAX_PROMPT, "tokens | worst-case sequence:", WORST_CASE_LEN, "tokens")


@torch.no_grad()
def generate_batch(student, prompts, eos_ids):
    """On-policy answers for several prompts at once (left padding)."""
    student.eval()
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    width = max(len(p) for p in prompts)
    input_ids = torch.full((len(prompts), width), pad_id, dtype=torch.long, device=STUDENT_DEVICE)
    attention = torch.zeros_like(input_ids)
    for i, p in enumerate(prompts):
        input_ids[i, width - len(p):] = torch.tensor(p, device=STUDENT_DEVICE)
        attention[i, width - len(p):] = 1
    with torch.autocast("cuda", dtype=HALF):
        out = student.generate(input_ids=input_ids, attention_mask=attention, max_new_tokens=MAX_NEW_TOKENS,
                               pad_token_id=pad_id, use_cache=True, **GEN_KWARGS)
    responses = []
    for row in out[:, width:].tolist():
        cut = len(row)
        for j, tok in enumerate(row):
            if tok in eos_ids:   # keep the end token, drop padding after it
                cut = j + 1
                break
        responses.append(row[:cut])
    return responses


def memory_probe(student, optimizer, vocab):
    """One forward/backward at the longest possible sequence, plus AdamW state allocation.
    If this fits, every shorter sequence fits too. Weights are not changed."""
    torch.cuda.reset_peak_memory_stats(STUDENT_DEVICE)
    torch.cuda.reset_peak_memory_stats(TEACHER_DEVICE)
    student.train()
    dummy = torch.randint(0, 1000, (1, WORST_CASE_LEN), device=STUDENT_DEVICE)
    loss = distill_loss(student, dummy, MAX_PROMPT, vocab)
    loss.backward()
    del loss, dummy
    # Allocate AdamW moments now (normally created at the first step) with lr=0, then reset them
    saved_lrs = [g["lr"] for g in optimizer.param_groups]
    for g in optimizer.param_groups:
        g["lr"] = 0.0
    optimizer.step()
    for g, lr in zip(optimizer.param_groups, saved_lrs):
        g["lr"] = lr
    for state in optimizer.state.values():
        for v in state.values():
            if torch.is_tensor(v):
                v.zero_()
    optimizer.zero_grad(set_to_none=True)
    total = torch.cuda.get_device_properties(STUDENT_DEVICE).total_memory
    peak0 = torch.cuda.max_memory_allocated(STUDENT_DEVICE)
    print(f"memory probe OK | student GPU peak {gb(peak0)} / {gb(total)} GB "
          f"(headroom {gb(total - peak0)} GB) | teacher GPU peak {gb(torch.cuda.max_memory_allocated(TEACHER_DEVICE))} GB")
    return gb(peak0)


def run_arm(lmbda):
    torch.cuda.reset_peak_memory_stats(STUDENT_DEVICE)
    student = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=torch.float32).to(STUDENT_DEVICE)
    student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    vocab = min(student.get_output_embeddings().weight.shape[0], teacher.get_output_embeddings().weight.shape[0])
    eos = student.generation_config.eos_token_id
    eos_ids = set(eos if isinstance(eos, list) else [eos]) | {tokenizer.convert_tokens_to_ids(END_TOKEN)}

    acc_before = quick_eval(student, STUDENT_DEVICE)

    optimizer = torch.optim.AdamW(student.parameters(), lr=LR)
    probe_peak = memory_probe(student, optimizer, vocab)
    scaler = torch.amp.GradScaler("cuda")

    order = list(range(N_TRAIN))
    random.shuffle(order)
    pos = 0
    log, oom_steps = [], 0

    for step in range(STEPS):
        t_step = time.time()
        batch = [train_set[order[(pos + k) % N_TRAIN]] for k in range(ACCUM)]
        pos += ACCUM
        prompts = [prompt_ids(ex["question"]) for ex in batch]
        on_policy = [random.random() < lmbda for _ in batch]

        # Responses: on-policy ones generated together in one batch, off-policy ones from the dataset
        responses = [None] * ACCUM
        gen_time = 0.0
        gen_idx = [i for i, f in enumerate(on_policy) if f]
        if gen_idx:
            t0 = time.time()
            for i, r in zip(gen_idx, generate_batch(student, [prompts[i] for i in gen_idx], eos_ids)):
                responses[i] = r
            gen_time = time.time() - t0
        for i, f in enumerate(on_policy):
            if not f:
                responses[i] = tokenizer(clean_solution(batch[i]["answer"]) + END_TOKEN,
                                         add_special_tokens=False)["input_ids"]

        # Loss one sequence at a time (memory stays bounded by the longest single sequence)
        optimizer.zero_grad(set_to_none=True)
        student.train()
        losses, resp_lens, skipped, oom = [], [], 0, False
        for p_ids, r_ids in zip(prompts, responses):
            if len(r_ids) == 0:
                skipped += 1
                continue
            loss, full = None, None
            try:
                full = torch.tensor([p_ids + r_ids], device=STUDENT_DEVICE)
                loss = distill_loss(student, full, len(p_ids), vocab)
                if not torch.isfinite(loss):
                    skipped += 1
                    continue
                scaler.scale(loss / ACCUM).backward()
                losses.append(loss.item())
                resp_lens.append(len(r_ids))
            except torch.cuda.OutOfMemoryError:
                oom = True
                break
            finally:
                del loss, full

        if oom:   # safety net: drop this step, free memory, keep training
            oom_steps += 1
            optimizer.zero_grad(set_to_none=True)
            gc.collect()
            torch.cuda.empty_cache()
            print(f"step {step + 1}: out of memory -> step skipped")
            continue

        if losses:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

        entry = {
            "step": step + 1,
            "loss": round(sum(losses) / max(len(losses), 1), 4),
            "step_seconds": round(time.time() - t_step, 2),
            "generation_seconds": round(gen_time, 2),
            "avg_response_tokens": round(sum(resp_lens) / max(len(resp_lens), 1), 1),
            "skipped": skipped,
            "vram_gpu0_gb": vram_used_gb(STUDENT_DEVICE),
            "vram_gpu1_gb": vram_used_gb(TEACHER_DEVICE),
        }
        log.append(entry)
        print(entry)

    acc_after = quick_eval(student, STUDENT_DEVICE)
    timed = log[1:] if len(log) > 1 else log   # first step includes warm-up
    result = {
        "lmbda": lmbda,
        "mode": "on-policy" if lmbda == 1.0 else ("off-policy" if lmbda == 0.0 else "mixed"),
        "steps": STEPS,
        "accum": ACCUM,
        "oom_steps": oom_steps,
        "probe_peak_gpu0_gb": probe_peak,
        "avg_step_seconds": round(sum(e["step_seconds"] for e in timed) / max(len(timed), 1), 2),
        "avg_generation_seconds_per_step": round(sum(e["generation_seconds"] for e in timed) / max(len(timed), 1), 2),
        "vram_gpu0_peak_pytorch_gb": gb(torch.cuda.max_memory_allocated(STUDENT_DEVICE)),
        "vram_gpu1_gb": vram_used_gb(TEACHER_DEVICE),
        f"quick_eval_acc_before_n{N_EVAL}": acc_before,
        f"quick_eval_acc_after_n{N_EVAL}": acc_after,
        "log": log,
    }

    del student, optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return result

longest prompt: 167 tokens | worst-case sequence: 551 tokens


## Run both arms

In [9]:
results = []
for lmbda in ARMS:
    print(f"\n===== LMBDA = {lmbda} =====")
    results.append(run_arm(lmbda))


===== LMBDA = 0.0 =====


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

memory probe OK | student GPU peak 11.12 / 14.56 GB (headroom 3.44 GB) | teacher GPU peak 8.09 GB
{'step': 1, 'loss': 0.3905, 'step_seconds': 2.32, 'generation_seconds': 0.0, 'avg_response_tokens': 137.2, 'skipped': 0, 'vram_gpu0_gb': 11.46, 'vram_gpu1_gb': 8.35}
{'step': 2, 'loss': 0.3837, 'step_seconds': 1.97, 'generation_seconds': 0.0, 'avg_response_tokens': 109.0, 'skipped': 0, 'vram_gpu0_gb': 11.46, 'vram_gpu1_gb': 8.36}
{'step': 3, 'loss': 0.4024, 'step_seconds': 2.01, 'generation_seconds': 0.0, 'avg_response_tokens': 86.0, 'skipped': 0, 'vram_gpu0_gb': 11.46, 'vram_gpu1_gb': 8.36}
{'step': 4, 'loss': 0.4176, 'step_seconds': 1.99, 'generation_seconds': 0.0, 'avg_response_tokens': 69.0, 'skipped': 0, 'vram_gpu0_gb': 11.46, 'vram_gpu1_gb': 8.36}
{'step': 5, 'loss': 0.2756, 'step_seconds': 2.08, 'generation_seconds': 0.0, 'avg_response_tokens': 101.2, 'skipped': 0, 'vram_gpu0_gb': 11.46, 'vram_gpu1_gb': 8.36}
{'step': 6, 'loss': 0.4515, 'step_seconds': 2.05, 'generation_seconds': 0.

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

memory probe OK | student GPU peak 11.12 / 14.56 GB (headroom 3.44 GB) | teacher GPU peak 8.09 GB
{'step': 1, 'loss': 0.246, 'step_seconds': 11.94, 'generation_seconds': 9.77, 'avg_response_tokens': 157.8, 'skipped': 0, 'vram_gpu0_gb': 11.46, 'vram_gpu1_gb': 8.36}
{'step': 2, 'loss': 0.2453, 'step_seconds': 23.22, 'generation_seconds': 20.46, 'avg_response_tokens': 323.0, 'skipped': 0, 'vram_gpu0_gb': 11.62, 'vram_gpu1_gb': 8.38}
{'step': 3, 'loss': 0.2862, 'step_seconds': 20.63, 'generation_seconds': 18.12, 'avg_response_tokens': 259.8, 'skipped': 0, 'vram_gpu0_gb': 11.62, 'vram_gpu1_gb': 8.38}
{'step': 4, 'loss': 0.205, 'step_seconds': 21.89, 'generation_seconds': 19.37, 'avg_response_tokens': 263.5, 'skipped': 0, 'vram_gpu0_gb': 11.62, 'vram_gpu1_gb': 8.38}
{'step': 5, 'loss': 0.2602, 'step_seconds': 23.47, 'generation_seconds': 20.54, 'avg_response_tokens': 330.2, 'skipped': 0, 'vram_gpu0_gb': 11.64, 'vram_gpu1_gb': 8.39}
{'step': 6, 'loss': 0.1617, 'step_seconds': 23.27, 'generati

## Summary and time estimate for a full run

In [10]:
FULL_TRAIN_QUESTIONS = 7473   # whole GSM8K train set, one epoch

for r in results:
    steps_full = FULL_TRAIN_QUESTIONS / ACCUM
    hours = steps_full * r["avg_step_seconds"] / 3600
    print(f"{r['mode']:>10}: {r['avg_step_seconds']} s/step "
          f"(generation {r['avg_generation_seconds_per_step']} s) | "
          f"peak student VRAM {r['vram_gpu0_peak_pytorch_gb']} GB | "
          f"quick eval {r[f'quick_eval_acc_before_n{N_EVAL}']} -> {r[f'quick_eval_acc_after_n{N_EVAL}']} | "
          f"one full epoch ~ {hours:.1f} h")

config = dict(teacher=TEACHER, student=STUDENT, n_train=N_TRAIN, steps=STEPS, accum=ACCUM, lr=LR,
              kl_direction=KL_DIRECTION, kd_temperature=KD_TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS,
              gen_kwargs=GEN_KWARGS, half_dtype=str(HALF), gpu=torch.cuda.get_device_name(0))
out_file = Path("/kaggle/working/kd_smoke_test.json")
out_file.write_text(json.dumps({"config": config, "results": results}, indent=2))
print("Saved to", out_file)

off-policy: 2.08 s/step (generation 0.0 s) | peak student VRAM 11.12 GB | quick eval 0.55 -> 0.45 | one full epoch ~ 1.1 h
 on-policy: 21.94 s/step (generation 19.31 s) | peak student VRAM 11.12 GB | quick eval 0.55 -> 0.5 | one full epoch ~ 11.4 h
Saved to /kaggle/working/kd_smoke_test.json
